## Importing libraries

In [ ]:
import numpy as np
import pandas as pd

import json

import glob
import os
from pathlib import Path

import pickle

## Importing data

In [ ]:
# Dictionary containing all the json files imported (json_file_name : json_content)
json_files_dict = {}

for path in sorted(Path('../raw_data-originals/dati_eumaco_con_pseudo').glob('*.json')):
    try:
        filename = path.stem
        json_content = json.loads(path.read_text()) # Closes automatically the file
        json_files_dict[filename] = json_content
    except json.JSONDecodeError:
        print(f"Skipping invalid JSON: {path}")

# List of all the json files names
json_filenames = list(json_files_dict)

## Make a pretty-print version of all the jsons

In [ ]:
# def pretty_print_folder(input_dir, output_dir):
#     os.makedirs(output_dir, exist_ok=True)

#     for filename in os.listdir(input_dir):
#         with open(os.path.join(input_dir, filename), "r", encoding="utf-8") as file:
#             data = json.load(file)

#         name = filename[:-5]  # strip ".json"
#         out_path = os.path.join(output_dir, f"{name}-pp.json")

#         with open(out_path, "w", encoding="utf-8") as file:
#             json.dump(data, file, indent=2, ensure_ascii=False, sort_keys=False)

In [ ]:
# input_dir = '../raw_data-originals/dati_eumaco_con_pseudo'
# output_dir = '../raw_data/jsons-pp'

# pretty_print_folder(input_dir, output_dir)

### Put specific values of the json in a pandas df

In [ ]:
def get_deep(dic, path):
    '''
    Helper function that takes in input:
    - a dictionary
    - a path to a value of the dictionary, colon separated
    and returns the corresponding value, even if the key:value pair is nested inside the dictionary

    For example, running the function on `d = {'ext': {'int': 'val'}}`:
    - get_deep(d, 'ext') -> {'int': 'val'}
    - get_deep(d, 'ext:int') -> 'val'
    '''
    
    current = dic
    for key in path.split(':'):
        if isinstance(current, dict) and key in current:
            current = current[key]
        else:
            return None
    return current

In [ ]:
def json_to_df(json_files_dict, json_filenames, *, just_external=False, transm_field=None, external_json_fields= None, transm_inst_fields=None):
    '''
    Takes in input:
    - a dictionary {json_filename: json_file_content}
    - a list of json filenames
    - the just_external bool option
    - the transmission subfield to check
    - the external fields common to every json file
    - the fields internal to the transmission subfield to check

    Iterates over the json files and the transmission field instances

    Returns a list containing dictionaries, where each one carries the informations
    of every instance of a specific transmission field,
    merged with the informations of the json itself

    If just_external is True, returns just the external informations of the json
    (number of rows will be equal to the number of json files)
    '''

    data = []

    # Iterates over the jsons
    for json_name in json_filenames:
        
        # current_json is 
        current_json = json_files_dict[json_name]

        # Dictionary containing all the "external" information of a json,
        # common to every instance of a transmission
        helper_dict = {}

        # Iterates over a list of external fields
        for field in external_json_fields:

            # Gets the value of the required field ...
            value = get_deep(current_json, field)
            # and puts it into the helper_dict
            helper_dict[field] = value

        # Return only the fields external to the transmission field
        if just_external == True:
            # Appends the data to the initial list
            data.append(helper_dict)
            continue

        # TODO: Maybe is better to place NAs?
        if transm_field not in current_json['transmissions']:
            continue

        # list containing all the instances of a transmission field
        instances = current_json['transmissions'][transm_field][0]

        for instance in instances:
            # Makes a copy of the common dictionary,
            # to allow to replace it at every loop
            instance_helper_dict = helper_dict
            
            # Iterates over a list of internal transmission fields
            for field in transm_inst_fields:

               # Gets the value of the required field ... 
                value = get_deep(instance, field)
                # and puts it into the instance_helper_dict
                instance_helper_dict[field] = value

                # Appends the data to the initial list
                data.append(instance_helper_dict)

    return data

In [ ]:
external_fields = [
    'patient_cf' ,
    'assignment_id' ,
    'archivied' ,
    'completed' ,

    'assignment:id' ,
    'assignment:start_at' ,
    'assignment:end_at' ,
    'assignment:active' ,
    'assignment:created_at' ,
    'assignment:updated_at' ,
    'assignment:hub_sn' ,
    'assignment:completed' ,
]

instance_ext_fields = [
    'id' ,
    'type' ,
    'report' ,
    'created_at' ,
    'updated_at' ,
    'registration_time' ,
    'device_sn' ,

# transmission_data
    'transmission_data:device' ,
    'transmission_data:serialnumber' ,
    'transmission_data:timestamp' ,
# battery
    'transmission_data:battery:value' ,
    'transmission_data:battery:unit' ,
# spo2
    'transmission_data:spo2:value' ,
    'transmission_data:spo2:unit' ,
# heartrate
    'transmission_data:heartrate:value' ,
    'transmission_data:heartrate:unit' ,
]

In [ ]:
data = json_to_df(json_files_dict, json_filenames,
    just_external = True,
    transm_field = 'SpO2Transmission',
    external_json_fields = external_fields,
    transm_inst_fields = instance_ext_fields
)
df = pd.DataFrame(data)

In [ ]:
df.info()

## Analysis and overview

### Check if the "external" fields except the `transmissions` have the same structure

In [ ]:
def columns_consistency(df, column1, column2, *, bidirectional=True, return_number=False):
    '''
    Finds inconsistency between two columns.
    Options:
    - if bidirectional is True (default), checks inconsistencies both between column1-column2 and column2-column1
    - if return_number is True, return only the number of inconsistent rows
    '''

    # Returns a Series where:
    # - index is the list of all the uniqued values of column1
    # - values are the most common column2 value for each column1 value
    mode_mapping_12 = df.groupby(column1)[column2].agg(lambda x: x.mode()[0])

    # Returns a Series where:
    # - index is the original index of df
    # - values are the most common column2 value for the column1 value
    expected_values_12 = df[column1].map(mode_mapping_12)

    # Makes a mask where every row that in column2 has a different expected value is marked as True
    mask = df[column2] != expected_values_12

    if bidirectional == True:
        # Makes a copy of the original mask, since later it will be overrided
        mask_12 = mask

        # Same operations than before
        mode_mapping_21 = df.groupby(column2)[column1].agg(lambda x: x.mode()[0])
        expected_values_21 = df[column2].map(mode_mapping_21)
        mask_21 = df[column1] != expected_values_21

        # Element-wise logical OR between the two masks
        mask = mask_12 | mask_21
    
    # Returns only the two involved columns and the rows where inconsistencies are found
    outliers = df[mask][[column1, column2]]

    if return_number == True:
        return f'Number of inconsistent rows: {len(outliers)}'

    return outliers

#### assignment_ids

In [ ]:
column1 = 'assignment_id'
column2 = 'assignment:id'
print(columns_consistency(df, column1, column2, return_number=True))

#### archived/completed

In [ ]:
column1 = 'archivied'
column2 = 'completed'

print(columns_consistency(df, column1, column2, return_number=True))

## Parsing and reformatting

Shows a sample of the field where the useful data is stored 

In [ ]:
json_files_dict[json_filenames[0]]['transmissions']['SpO2Transmission'][0][0]

### Put all the `patient_cf` fields in a list

In [ ]:
# ids = []

# for json in json_files:
#     ids.append(json['patient_cf'])

In [ ]:
# ser = pd.Series(ids)
# ser

## Export

In [ ]:
# # Pickle
# with open('../raw_data/pickles/4-eumaco.pickle', 'wb') as file:
#     pickle.dump(df, file, protocol=pickle.HIGHEST_PROTOCOL)